# 08. 최종 예측모델 선정 (Validation 기준) - LightGBM vs XGBoost

## 목적

기존 05_2차_모델링_LightGBM_ML1조.ipynb / 06_3차_모델링_XGBOOST_ML1조.ipynb는 Validation(2024)을 early stopping에만
사용하고, LightGBM vs XGBoost 비교 및 최종 모델 선정은 Test(2025~2026) 지표로 해왔다.

통계적으로는 Train = 학습 / Validation = 모델·하이퍼파라미터 선택 / Test = 최종 일반화 성능 평가가 맞으므로,
이 노트북에서는:

1. LightGBM, XGBoost를 05/06번과 동일한 피처·하이퍼파라미터로 재학습(재현)한다.
2. Validation(2024) R²/RMSE/MAE를 나란히 비교해 최종 예측모델을 선정한다.
3. Test(2025~2026)는 선정에 관여하지 않고, 선정된 모델의 일반화 성능을 확인하는 용도로만 마지막에 제시한다.
4. Validation과 Test의 우위 모델이 다르더라도 선정은 Validation 기준을 유지하고, 그 차이는
   "모델별 일반화 성능 차이"로 해석한다 (Test를 보고 모델을 다시 바꾸지 않음).


# 0. 데이터 업로드

활용 데이터셋: dataset_final_ML1조.csv (05/06번과 동일)

In [ ]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from google.colab import files
print("dataset_final_ML1조 파일 업로드")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

# Train(2023) = 학습 전용 / Val(2024) = 모델 선정 기준 / Test(2025~2026) = 선정 이후 일반화 성능 확인(1회)
train_test_split_raw = df[df['data_split'] == 'train'].copy()
test_df = df[df['data_split'] == 'test'].copy()

train_df = train_test_split_raw[train_test_split_raw['계약연도'] == 2023].copy()
val_df = train_test_split_raw[train_test_split_raw['계약연도'] == 2024].copy()

print(f"Train(2023)     : {train_df.shape[0]}건")
print(f"Val(2024)       : {val_df.shape[0]}건")
print(f"Test(2025~2026) : {test_df.shape[0]}건")


In [ ]:
target = '면적당가격_만원'
features_gbm = [
    'log_전용면적', '층', '주택연령', 'is_basement',
    'SK미래관거리', '보문역거리',
    '면적라벨_소형', '면적라벨_중형',
    '전월세구분_전세', '동그룹_안암동', '주택유형_연립'
]

X_train = train_df[features_gbm]
y_train = train_df[target]
X_val = val_df[features_gbm]
y_val = val_df[target]
X_test = test_df[features_gbm]
y_test = test_df[target]


# 1. LightGBM 재학습 (05번과 동일 하이퍼파라미터)

In [ ]:
lgb_model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=10,
    random_state=42
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

y_val_pred_lgb = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration_)
y_test_pred_lgb = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration_)

metrics_lgb = {
    'model': 'LightGBM',
    'val_r2': r2_score(y_val, y_val_pred_lgb),
    'val_rmse': np.sqrt(mean_squared_error(y_val, y_val_pred_lgb)),
    'val_mae': mean_absolute_error(y_val, y_val_pred_lgb),
    'test_r2': r2_score(y_test, y_test_pred_lgb),
    'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred_lgb)),
    'test_mae': mean_absolute_error(y_test, y_test_pred_lgb),
    'best_iteration': int(lgb_model.best_iteration_),
}
print(metrics_lgb)


# 2. XGBoost 재학습 (06번과 동일 하이퍼파라미터)

In [ ]:
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=50,
    eval_metric='rmse',
    importance_type='gain',
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_val_pred_xgb = xgb_model.predict(X_val, iteration_range=(0, xgb_model.best_iteration + 1))
y_test_pred_xgb = xgb_model.predict(X_test, iteration_range=(0, xgb_model.best_iteration + 1))

metrics_xgb = {
    'model': 'XGBoost',
    'val_r2': r2_score(y_val, y_val_pred_xgb),
    'val_rmse': np.sqrt(mean_squared_error(y_val, y_val_pred_xgb)),
    'val_mae': mean_absolute_error(y_val, y_val_pred_xgb),
    'test_r2': r2_score(y_test, y_test_pred_xgb),
    'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred_xgb)),
    'test_mae': mean_absolute_error(y_test, y_test_pred_xgb),
    'best_iteration': int(xgb_model.best_iteration),
}
print(metrics_xgb)


# 3. Validation(2024) 기준 최종 모델 선정

In [ ]:
val_compare = pd.DataFrame([
    {'Model': metrics_lgb['model'], 'Val_R2': metrics_lgb['val_r2'], 'Val_RMSE': metrics_lgb['val_rmse'], 'Val_MAE': metrics_lgb['val_mae']},
    {'Model': metrics_xgb['model'], 'Val_R2': metrics_xgb['val_r2'], 'Val_RMSE': metrics_xgb['val_rmse'], 'Val_MAE': metrics_xgb['val_mae']},
])
print("=" * 60)
print(" Validation(2024) 기준 모델 비교 - 최종 예측모델 선정")
print("=" * 60)
print(val_compare.to_string(index=False))

best_model = val_compare.loc[val_compare['Val_R2'].idxmax(), 'Model']
print(f"\nValidation R2 기준 최종 선정 모델: {best_model}")


# 4. (참고) Test(2025~2026) 일반화 성능 확인 - 선정 기준 아님

In [ ]:
test_compare = pd.DataFrame([
    {'Model': metrics_lgb['model'], 'Test_R2': metrics_lgb['test_r2'], 'Test_RMSE': metrics_lgb['test_rmse'], 'Test_MAE': metrics_lgb['test_mae']},
    {'Model': metrics_xgb['model'], 'Test_R2': metrics_xgb['test_r2'], 'Test_RMSE': metrics_xgb['test_rmse'], 'Test_MAE': metrics_xgb['test_mae']},
])
print("=" * 60)
print(" (참고) Test(2025~2026) 성능 - 선정 모델의 일반화 성능 확인용, 선정 기준 아님")
print("=" * 60)
print(test_compare.to_string(index=False))

best_test = test_compare.loc[test_compare['Test_R2'].idxmax(), 'Model']
if best_model != best_test:
    print(f"\n[주의] Validation 우위 모델({best_model})과 Test 우위 모델({best_test})이 다릅니다.")
    print("  선정은 Validation 기준을 유지하고, 이 차이는 모델별 일반화 성능 차이로 해석한다.")
else:
    print(f"\nValidation과 Test 모두 {best_model}이 우위 - 선정 결과가 일반화 성능에서도 일관되게 확인됨.")

# 최종 metrics 저장 (보고서/다른 노트북에서 재사용)
with open('final_model_selection_metrics.json', 'w', encoding='utf-8') as f:
    json.dump({'lightgbm': metrics_lgb, 'xgboost': metrics_xgb, 'selected_model': best_model}, f, ensure_ascii=False, indent=2)

from google.colab import files
files.download('final_model_selection_metrics.json')
